# Analysing both the datasets

In [99]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df1 = pd.read_csv('./dataset1_cleaned.csv', parse_dates=True)
df2 = pd.read_csv('./dataset2_cleaned.csv', parse_dates=True)


In [100]:
df1.head()

,start_time,bat_landing_to_food_min,habit,rat_period_start,rat_period_end,min_after_rat_arrival,risk,reward,month,sunset_time,min_after_sunset,season,date,chronology_check
0,2017-12-26 20:57:00,0.016667,Unknown,2017-12-26 20:53:00,2017-12-26 20:58:00,3.983333,0,0,0,2017-12-26 16:43:00,254.916667,0,2017-12-26,True
1,2017-12-26 20:57:00,0.083333,Unknown,2017-12-26 20:53:00,2017-12-26 20:58:00,3.316667,0,0,0,2017-12-26 16:43:00,254.250000,0,2017-12-26,True
2,2017-12-26 21:24:00,0.050000,fast,2017-12-26 21:22:00,2017-12-26 21:27:00,2.016667,0,1,0,2017-12-26 16:43:00,281.616667,0,2017-12-26,True
3,2017-12-26 21:24:00,0.250000,rat,2017-12-26 21:22:00,2017-12-26 21:27:00,1.466667,1,0,0,2017-12-26 16:43:00,281.066667,0,2017-12-26,True
4,2017-12-26 21:24:00,0.100000,pick,2017-12-26 21:22:00,2017-12-26 21:27:00,1.883333,0,1,0,2017-12-26 16:43:00,281.483333,0,2017-12-26,True


In [101]:
df2.head()

,time,month,bat_landing_number,food_availability,rat_minutes,rat_arrival_number,minutes_after_sunset,date
0,2017-12-26 16:13:00,0,20,4.000000,0.0,0,-30.0,2017-12-26
1,2017-12-26 16:43:00,0,28,4.000000,0.0,0,0.0,2017-12-26
2,2017-12-26 17:13:00,0,25,4.000000,0.0,0,30.0,2017-12-26
3,2017-12-26 17:43:00,0,71,4.000000,0.0,0,60.0,2017-12-26
4,2017-12-26 18:13:00,0,44,3.753857,0.0,0,90.0,2017-12-26


### Calculating Number of Bat landing per 30min time interval
bat_landing_number in dataset2 contains the sequence for individual bat landing. So, we are now grouping the bat landing with interval_time then counting bats per 30-min interval from dataset 1.

In [102]:
df1['start_time'] = pd.to_datetime(df1['start_time'])
df1['interval_time'] = df1['start_time'].dt.floor('30min')

# Counting bats per 30 min
bat_count = (df1.groupby(['date', 'interval_time']).size().rename('bat_count').reset_index())

# Merging that count back to each bat row 
df1 = df1.merge(bat_count, on=['date', 'interval_time'], how='left')
df1.head()

,start_time,bat_landing_to_food_min,habit,rat_period_start,rat_period_end,min_after_rat_arrival,risk,reward,month,sunset_time,min_after_sunset,season,date,chronology_check,interval_time,bat_count
0,2017-12-26 20:57:00,0.016667,Unknown,2017-12-26 20:53:00,2017-12-26 20:58:00,3.983333,0,0,0,2017-12-26 16:43:00,254.916667,0,2017-12-26,True,2017-12-26 20:30:00,2
1,2017-12-26 20:57:00,0.083333,Unknown,2017-12-26 20:53:00,2017-12-26 20:58:00,3.316667,0,0,0,2017-12-26 16:43:00,254.250000,0,2017-12-26,True,2017-12-26 20:30:00,2
2,2017-12-26 21:24:00,0.050000,fast,2017-12-26 21:22:00,2017-12-26 21:27:00,2.016667,0,1,0,2017-12-26 16:43:00,281.616667,0,2017-12-26,True,2017-12-26 21:00:00,7
3,2017-12-26 21:24:00,0.250000,rat,2017-12-26 21:22:00,2017-12-26 21:27:00,1.466667,1,0,0,2017-12-26 16:43:00,281.066667,0,2017-12-26,True,2017-12-26 21:00:00,7
4,2017-12-26 21:24:00,0.100000,pick,2017-12-26 21:22:00,2017-12-26 21:27:00,1.883333,0,1,0,2017-12-26 16:43:00,281.483333,0,2017-12-26,True,2017-12-26 21:00:00,7


# Joining Two Datasets
W

In [103]:
# Ensuring both the column are in datetime format and sorting 
df1['start_time'] = pd.to_datetime(df1['start_time']).sort_values(ascending=True)
df2['time'] = pd.to_datetime(df2['time']).sort_values(ascending=True)

# Merging df1 and df2 on date
# data = pd.merge(df1,df2, left_on='date', right_on='date', how='left')
#  merge ran into row explosion

In [104]:
# Merging df1 and df2 
data = pd.merge_asof(df1, df2, left_on='start_time', right_on='time', direction='backward')

In [105]:
data.info()
data.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 906 entries, 0 to 905
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   start_time               906 non-null    datetime64[ns]
 1   bat_landing_to_food_min  906 non-null    float64       
 2   habit                    906 non-null    object        
 3   rat_period_start         906 non-null    object        
 4   rat_period_end           906 non-null    object        
 5   min_after_rat_arrival    906 non-null    float64       
 6   risk                     906 non-null    int64         
 7   reward                   906 non-null    int64         
 8   month_x                  906 non-null    int64         
 9   sunset_time              906 non-null    object        
 10  min_after_sunset         906 non-null    float64       
 11  season                   906 non-null    int64         
 12  date_x                   906 non-nul

start_time                 0
bat_landing_to_food_min    0
habit                      0
rat_period_start           0
rat_period_end             0
min_after_rat_arrival      0
risk                       0
reward                     0
month_x                    0
sunset_time                0
min_after_sunset           0
season                     0
date_x                     0
chronology_check           0
interval_time              0
bat_count                  0
time                       0
month_y                    0
bat_landing_number         0
food_availability          0
rat_minutes                0
rat_arrival_number         0
minutes_after_sunset       0
date_y                     0
dtype: int64

# Confirming the rat presence during bat landing
In this section,we are confirming if there was rat on food platform when bat landed. 

In [106]:
# Method 1
# data['rat_present'] = (
#     (data['start_time'] >= data['rat_period_start']) &
#     (data['start_time'] <= data['rat_period_end'])
# )
# # data['rat_present'] = data['rat_present'].astype(int)


# Method 2
data['rat_present'] = data['rat_minutes'].fillna(0) > 0
data['rat_present'].unique()

array([False,  True])

# Calculating Bat Hesitation (time to food)

In [107]:
avg_time_to_food = data.groupby('rat_present')['bat_landing_to_food_min'].mean()
avg_time_to_food

rat_present
False    0.092347
True     0.066937
Name: bat_landing_to_food_min, dtype: float64

# Food Availability Unit
We have given food availability data but the unit is unkown. So we are truning given data into fraction of 4, so it's easy to interpret. Zero (0) represent no food and 1 represent food. 

In [108]:
data['food_fraction'] = (data['food_availability'] / 4).round(2)
data['food_fraction']

0      0.74
1      0.74
2      0.64
3      0.64
4      0.64
       ... 
901    0.50
902    0.50
903    0.50
904    0.50
905    0.75
Name: food_fraction, Length: 906, dtype: float64

# Calculating Risk Level
Now we have risk taken behavior and food availability unit. We are calculating risk taken level for bat. We want to identify if bat took more risk when number of rat is greater than number of bat or vice versa. Similarly, availability of food during the risk taken.

### Comparing number of rats vs number of bats

In [109]:
data['rats_more_than_bats'] = (data['rat_arrival_number'] > data['bat_count']).astype(int)
data['rats_more_than_bats'].unique()

array([0, 1])

### Compare risk vs food availability

In [110]:
data.groupby('risk')['food_fraction'].mean()

risk
0    0.628166
1    0.641295
Name: food_fraction, dtype: float64

### Combining both rat and food context

In [111]:
summary = data.groupby(['risk', 'rats_more_than_bats']).agg(
    avg_food_fraction = ('food_fraction', 'mean'),
    avg_bat_to_food = ('bat_landing_to_food_min', 'mean'),
    count = ('start_time', 'count')
)
summary

avg_food_fraction  avg_bat_to_food  count
risk rats_more_than_bats                                           
0    0                             0.626532         0.068040    447
     1                             0.694545         0.095727     11
1    0                             0.641164         0.115897    438
     1                             0.647000         0.128401     10